# MAPPO Curriculum

Train the first forage curriculum stage for `AntByteForagingEnv`: ants learn to reach cookie sources, pick up bites, return to the hub, and write tile values. The notebook runs from `4x4` through `25x25`, but the reusable training and rendering code lives in `ant_byte_env.notebook_workflows`.


In [ ]:
from pathlib import Path
import os
import sys

# Set these before importing JAX in this kernel.
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.35")
if "jax" in sys.modules:
    print("Restart the kernel before rerunning training; JAX was already imported.")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from ant_byte_env import notebook_workflows as workflows

runtime_status = workflows.configure_jax_notebook_runtime()
workflows.assert_notebook_resources_available(runtime_status)
{"project_root": PROJECT_ROOT, **runtime_status}


In [ ]:
import importlib

import jax

from ant_byte_env import notebook_workflows as workflows
from ant_byte_env.training.jax_mappo import runner as jax_runner

workflows = importlib.reload(workflows)
jax_runner = importlib.reload(jax_runner)
print(f"JAX device: {jax.devices()[0]}")


## Quick Smoke Run

Run one tiny training job to confirm the kernel, package imports, and JAX path are wired correctly.


In [ ]:
smoke_metrics = workflows.run_jax_smoke(jax_runner.main)
smoke_metrics


## Curriculum Settings

Edit only the stage sizes or high-level run constants here. The stage construction and CLI argument plumbing are shared with the other notebooks.


In [ ]:
RUN_DIR = PROJECT_ROOT / "runs" / "notebooks" / "forage_curriculum"
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
MEDIA_DIR = RUN_DIR / "media"

STAGE_SIZES = (4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 25)
NUM_ENVS = 16
NUM_STEPS = 80
GLOBAL_UPDATE_CAP = 2000
ROLLOUT_TILE_SIZE = workflows.NOTEBOOK_ROLLOUT_TILE_SIZE
ACTOR_VISION_RADIUS = 1
WRITE_BITS = 1

CURRICULUM_STAGES = workflows.build_forage_curriculum_stages(STAGE_SIZES)
COMMON_ARGS = workflows.build_forage_common_args(
    CURRICULUM_STAGES,
    num_envs=NUM_ENVS,
    num_steps=NUM_STEPS,
    actor_vision_radius=ACTOR_VISION_RADIUS,
    write_bits=WRITE_BITS,
)
UPDATE_TIMESTEPS = workflows.update_timesteps(num_envs=NUM_ENVS, num_steps=NUM_STEPS)

{
    "stages": [stage["name"] for stage in CURRICULUM_STAGES],
    "updates_per_stage": GLOBAL_UPDATE_CAP,
    "update_timesteps": UPDATE_TIMESTEPS,
}


## Train Curriculum Checkpoints


In [ ]:
forage_result = workflows.run_forage_curriculum(
    stages=CURRICULUM_STAGES,
    checkpoint_dir=CHECKPOINT_DIR,
    common_args=COMMON_ARGS,
    update_timesteps_per_stage=UPDATE_TIMESTEPS,
    global_update_cap=GLOBAL_UPDATE_CAP,
    train_main=jax_runner.main,
)
FINAL_CHECKPOINT_PATH = forage_result["final_checkpoint_path"]
forage_result


## Optional Render and Vault


In [ ]:
rollout_result = workflows.render_forage_rollouts(
    run_dir=RUN_DIR,
    checkpoint_dir=CHECKPOINT_DIR,
    media_dir=MEDIA_DIR,
    stages=CURRICULUM_STAGES,
    actor_vision_radius=ACTOR_VISION_RADIUS,
    write_bits=WRITE_BITS,
    global_update_cap=GLOBAL_UPDATE_CAP,
    tile_size=ROLLOUT_TILE_SIZE,
)
rollout_result
